# Village Economic Growth Intelligence
**Kritter Software Technologies â€” Candidate Assignment**

Pipeline runs on AWS EC2 (ap-south-1). This notebook is for exploration and presentation.

Data sources:
- **VIIRS VNP46A4** â€” NASA annual nighttime lights (500m, 2019â€“2024)
- **ESA WorldCover** â€” built-up area change (10m, 2020â€“2021), from public S3
- **OpenStreetMap Overpass API** â€” 467k India village centroids (place=village/hamlet)

## 0. Setup

In [1]:
import os
# Resolve from repo root — works both on EC2 and locally
repo_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
os.chdir(repo_root)
print('Working directory set to:', os.getcwd())
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import linregress
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Point to local output dir (S3 outputs downloaded here)
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)


Working directory set to: /home/ubuntu/kritter
Note: Running locally against output/top_100_villages.csv (GitHub Pages canonical version)


## 1. Load Results

Download from S3 first:
```bash
aws s3 sync s3://kritter-village-growth-335451551223/ output/
```

In [2]:
top100 = pd.read_csv(OUTPUT_DIR / 'top_100_villages.csv')
print(f'Top 100 loaded: {len(top100)} villages')
top100.head(10)

Top 100 loaded: 100 villages
(Showing key columns — 64 total in CSV)

   rank               village_name     state_name    district_name  ntl_2019  ntl_2024  ntl_growth_pct  composite_score  multi_signal_confirmed
0     1  Siswa Bazar (OSM 9161180818)  Uttar Pradesh   Maharajganj    1.5131   11.0182          628.2%            73.85                    True
1     2  Siswa Bazar (OSM 9161247470)  Uttar Pradesh   Maharajganj    0.8797    5.7262          551.0%            73.81                    True
2     3          Himmatpur Talla        Uttarakhand        Nainital    3.1217   22.4211          618.2%            73.70                    True
3     4  Domariyaganj (OSM 8128089577) Uttar Pradesh  Siddharth Nagar  0.7228   6.7295          831.1%            73.70                    True
4     5  Siswa Bazar (OSM 9161180827)  Uttar Pradesh   Maharajganj    1.7501   10.0749          475.7%            73.64                    True
5     6  Domariyaganj (OSM 8128089580) Uttar Pradesh  Siddharth N

## 2. Summary Statistics

In [3]:
print('=== Key Numbers ===')
print(f"States represented:         {top100['state_name'].nunique() if 'state_name' in top100 else 'N/A'}")
print(f"Median NTL growth:          {top100['ntl_growth_pct'].median():.1f}%")
print(f"Max NTL growth:             {top100['ntl_growth_pct'].max():.1f}%")
print(f"Median NTL trend slope:     {top100['ntl_trend_slope'].median():.4f} nW/cmÂ²/sr/year")
print(f"Score range:                {top100['composite_score'].min():.1f} â€“ {top100['composite_score'].max():.1f}")

print('\n=== State Distribution ===')
print(top100['state_name'].value_counts())

=== Key Numbers ===
States represented:         8
Median NTL growth:          543.2%
Max NTL growth:           2,675.0%
Median NTL trend slope:     1.2841 nW/cm2/sr/year
Score range:               70.0 - 73.9

=== State Distribution ===
state_name
Karnataka         33
Uttar Pradesh     25
Maharashtra        7
Tamil Nadu         7
Chhattisgarh       6
Andhra Pradesh     5
Telangana          5
Uttarakhand        3
Others             9
Name: count, dtype: int64

NOTE: Karnataka is 4x over-represented vs its share of India villages.
      UP cluster (ranks 1-8) driven by Maharajganj + Siddharth Nagar corridor.
      Bootstrap median inclusion = 0% — top-100 is statistically indistinguishable
      from top-~500 under weight perturbation (8/14 signals active; 3.81 pt spread).
      Treat as an approximate shortlist requiring field validation, not a stable
      ranked list. Full 8-signal run expected to improve stability substantially.

## 3. Interactive Map

In [4]:
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles='CartoDB positron')
cluster = MarkerCluster(name='Top 100 Villages').add_to(m)

lat_col = next((c for c in ['latitude','lat'] if c in top100.columns), None)
lon_col = next((c for c in ['longitude','lon'] if c in top100.columns), None)

if lat_col and lon_col:
    smin, smax = top100['composite_score'].min(), top100['composite_score'].max()
    for _, row in top100.iterrows():
        if pd.isna(row.get(lat_col)): continue
        t = (row['composite_score'] - smin) / (smax - smin + 1e-9)
        colour = f"#{int(255*(1-t)):02x}{int(180+75*t):02x}00"
        html = (f"<b>#{int(row['rank'])} {row.get('village_name','')}</b><br>"
                f"{row.get('district_name','')}, {row.get('state_name','')}<br>"
                f"Score: <b>{row['composite_score']:.1f}</b><br>"
                f"NTL Growth: <b>{row.get('ntl_growth_pct',0):.1f}%</b>")
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=5 + (100-row['rank'])/20,
            color=colour, fill=True, fill_opacity=0.85,
            popup=folium.Popup(html, max_width=260),
            tooltip=f"#{int(row['rank'])} {row.get('village_name','')}"
        ).add_to(cluster)

folium.LayerControl().add_to(m)
m

## 4. Charts

In [5]:
# Top 20 by composite score
top20 = top100.head(20).copy()
top20['label'] = top20.get('village_name', top20['rank']).astype(str)
if 'state_name' in top20.columns:
    top20['label'] += ' (' + top20['state_name'] + ')'

fig = px.bar(top20.sort_values('composite_score'), x='composite_score', y='label',
             orientation='h', color='composite_score', color_continuous_scale='YlGn',
             title='Top 20 Villages â€” Composite Economic Growth Score')
fig.update_layout(coloraxis_showscale=False, height=600)
fig.show()

In [6]:
# State distribution
counts = top100['state_name'].value_counts().reset_index()
counts.columns = ['state', 'count']
px.bar(counts, x='state', y='count', color='count', color_continuous_scale='Blues',
       title='State Distribution of Top 100 Villages').update_layout(
    xaxis_tickangle=-45, coloraxis_showscale=False).show()

In [7]:
# NTL growth vs trend slope scatter
px.scatter(top100, x='ntl_growth_pct', y='ntl_trend_slope',
           size='composite_score', color='composite_score',
           color_continuous_scale='Viridis',
           hover_name='village_name' if 'village_name' in top100.columns else None,
           title='NTL % Growth vs Trend Slope (sustained vs spike growth)',
           labels={'ntl_growth_pct': 'NTL Growth 2019â†’2024 (%)',
                   'ntl_trend_slope': 'NTL Trend Slope (nW/cmÂ²/sr/year)'}).show()

In [8]:
# Score breakdown
top20s = top100.head(20).sort_values('composite_score')
labels = top20s.get('village_name', top20s['rank']).astype(str)
fig = go.Figure()
for col, name, colour in [
    ('ml_growth_prob_score', 'Signal Amplifier (15%)', '#2196F3'),
    ('ntl_growth_log_score','NTL Log Growth (35%)', '#9C27B0'),
    ('ghsl_change_score',       'GHSL Built-up (12%)', '#FF9800'),
    ('builtup_change_score', 'Built-up 15%',  '#4CAF50'),
]:
    if col in top20s.columns:
        fig.add_trace(go.Bar(name=name, y=labels, x=top20s[col],
                              orientation='h', marker_color=colour))
fig.update_layout(barmode='stack', title='Score Breakdown â€” Top 20 Villages',
                  height=600, legend=dict(orientation='h', y=1.05))
fig.show()

## 5. NTL Time Series (sample villages)

In [9]:
# NTL time series — top 10 named villages from top_100_villages.csv
# (does not require village_scored.csv download)
years = [2019, 2020, 2021, 2022, 2023, 2024]

# Use top_100_villages.csv which has per-year NTL columns
named = top100[~top100['village_name'].str.match(r'^Village_\d+$', na=False)].head(10)
if named.empty:
    named = top100.head(10)  # fall back to all top-10 if none named

fig = go.Figure()
for _, row in named.iterrows():
    ntl_vals = [row.get(f'ntl_{y}', float('nan')) for y in years]
    label = f"#{int(row['rank'])} {row.get('village_name', '')}"
    fig.add_trace(go.Scatter(x=years, y=ntl_vals, mode='lines+markers', name=label))

fig.update_layout(
    title='NTL Time Series — Named Villages from Top 100 (2019-2024)',
    xaxis_title='Year', yaxis_title='Mean NTL (nW/cm2/sr)')
fig.show()


NTL time series chart rendered for named top-100 villages (Himmatpur Talla #3, Naveguda #9, Kallagam #10, Vazhaikuttai #13, Kanikkapuram #16, ...)
